# 第3回：pandasで表データに触る

**今日の問い：初めて見る表データを受け取ったら、最初に何を見るか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 初見データの形・型・欠損・要約統計を確認する
- locとqueryで条件を明示し、method chainingで読みやすくまとめる
- groupby・agg・pivot_tableで多軸の比較表を作り、性能差にも気を配る

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- DataFrame：行と列を持つ表
- method chaining：中間変数を作らず処理をつなげる書き方
- ベクトル化：ループの代わりに列全体へ一括演算すること
- 集約：複数行を件数や平均などへまとめる処理
- カテゴリ型：取りうる値が限られる列の省メモリ表現

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## pandasは「表を操る道具」

pandasは、Excelのような表（DataFrame）をPythonで扱うライブラリです。研究データの多くは表なので、
これが読めると分析の8割は前に進みます。この回で身につけるのは、初見の表に対して**同じ手順で
最初の点検をする**習慣です。

初見データを受け取ったら、まず次の4つを見ます：**大きさ（行数×列数）／型（数値か文字か）／
欠損（空欄はどこか）／ばらつき（平均や範囲）**。名探偵が現場でまず全体を見渡すのと同じです。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## TRY：表の「健康診断」を1度に行う

次のセルは、点検の4項目をまとめて表示します。`df.shape`で大きさ、`df.dtypes`で型、
`df.isna()`で欠損、`df.describe()`で要約統計。**関数名がそのまま意味**なので、少しずつ覚えられます。


In [ ]:
print("形:", df.shape)
quality = pd.DataFrame({
    "データ型": df.dtypes.astype(str),
    "欠損数": df.isna().sum(),
    "欠損率": df.isna().mean().round(3),
    "ユニーク数": df.nunique(),
})
display(quality)
display(df.select_dtypes(include="number").describe().T.round(2))


### 出力の読み方

- **1つ目の表（品質表）**：各列の型・欠損数・欠損率・値の種類数。`object`は文字列、`float64`/`int64`は数値。欠損率が高い列や、`sample_id`のようにユニーク数＝行数の列（＝ただの名札）に気づけます。
- **2つ目の表（describe）**：数値列の件数・平均・標準偏差・最小/四分位/最大。`temperature_c`の`max`が極端に大きいなど、**怪しい値の当たり**をここで付けます。
- `.T`は表を**転置**（行列入れ替え）して、列がたくさんあっても縦に読めるようにする工夫です。


## 行と列を選ぶ：`loc` と `query`

分析は「必要な部分だけ取り出す」の連続です。2つの基本を覚えます。

- `df.loc[行の条件, 列のリスト]`：**場所を指定して取り出す**。
- `df.query("条件式")`：**条件を文章のように書いて絞り込む**。複数条件（`and`/`or`）が読みやすいのが利点です。


In [ ]:
columns = ["sample_id", "solvent", "catalyst", "temperature_c", "yield_pct", "active"]
display(df.loc[:4, columns])
subset = df.query("catalyst == 'Cat-A' and temperature_c >= 80")[columns]
print("Cat-Aかつ80℃以上:", len(subset), "件")
subset.head()


### 出力の読み方とつまずきポイント

- `df.loc[:4, columns]`は「行番号0〜4」×「指定した6列」。`loc`の範囲指定は**末尾を含む**点がPythonの通常のスライス（末尾を含まない）と違うので注意します。
- `query`の中では、文字列は`'Cat-A'`のように**引用符**で囲みます。列名はそのまま書けます。
- `len(subset)`で、条件に合った件数が分かります。**まず件数を確かめる**のは、絞り込みが意図どおりかの安全確認です。


## TRY：カテゴリごとにまとめて比べる（groupby）

「溶媒ごとの平均収率は？」のような問いには`groupby`が使えます。**同じ値の行をまとめて、
件数・平均・ばらつきなどを一気に計算**します。平均だけでなく**件数（size）とばらつき（std）**も
一緒に見るのが、だまされないコツです。


In [ ]:
solvent_summary = (
    df.groupby("solvent", dropna=False)
      .agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean"),
           収率SD=("yield_pct", "std"), 活性率=("active", "mean"))
      .sort_values("平均収率", ascending=False)
)
solvent_summary.round(2)


### 出力の読み方

- 溶媒ごとに1行、件数・平均収率・収率のばらつき・活性率が並び、平均収率の高い順に並びます。
- **平均が高くても件数が極端に少ない**溶媒は、たまたまかもしれません。件数の小さい行の平均は割り引いて読みます。
- `dropna=False`にしているので、溶媒が欠損の行も1グループとして見えます（欠損を見逃さない工夫）。


## CHANGE

`groupby("solvent")`を`"catalyst"`や`"scaffold_group"`へ変えて、順位がどう変わるか見ます。
順位が変わる理由は、**データだけから断定せず仮説として**書き留めます（第4〜5回でその検証を学びます）。


## DEEP DIVE：多軸集計・処理の連結・速度

発展として、実務でよく使う3つを扱います。**pivot_table**（2軸のクロス集計）、
**pipe**（処理を関数でつなぐ）、そして**ベクトル化**（速く書く）です。


### pivot_table：2つの軸で同時に集計する

「触媒×溶媒」のように2軸で平均を見たいときは`pivot_table`が便利です。Excelのピボットテーブルと
同じ発想で、`index`（縦軸）・`columns`（横軸）・`values`（集計する値）・`aggfunc`（集計方法）を指定します。


In [ ]:
pivot = pd.pivot_table(df, index="catalyst", columns="solvent", values="yield_pct", aggfunc=["count", "mean"])
pivot.round(1)


### 出力の読み方

行が触媒、列が溶媒で、各マスに「件数」と「平均収率」が入ります。件数が0や極端に少ないマスは、
平均が空欄や不安定になります。**組み合わせによって効き方が変わる**様子（交互作用）の当たりを付けられます。


### pipe：処理を「関数の流れ」としてつなぐ

複数の加工を続けるとき、中間変数を増やすと読みにくくなります。`.pipe(関数)`を使うと、
**表を関数に通して次へ渡す**流れを、上から下へ素直に書けます。元データを壊さないよう、関数内で`copy()`します。


In [ ]:
def add_quality_flags(frame):
    "収率の中央値以上かどうかのフラグ列を足して返す（元は変更しない）。"
    out = frame.copy()
    out["high_yield"] = out["yield_pct"] >= out["yield_pct"].median()
    return out

summary = (
    df
    .pipe(add_quality_flags)
    .groupby(["catalyst", "high_yield"], observed=True)
    .agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean"))
    .round(2)
)
summary


### 出力の読み方

触媒×「高収率かどうか」で件数と平均収率が出ます。**加工（フラグ付け）→集計**という流れが、
1つの縦長の式で読める点に注目してください。処理が増えても`.pipe(...)`を足すだけで拡張できます。


## CHALLENGE：`apply`と「ベクトル化」の速度差

同じ判定を2通りで書き、時間を比べます。行を1つずつ処理する`apply`は読みやすい一方、遅くなりがち。
列全体へ一括で演算する**ベクトル化**は速く、pandasの本領です。


In [ ]:
import time

def slow_flag(frame):
    return frame.apply(lambda r: r["temperature_c"] >= 80 and r["catalyst"] == "Cat-A", axis=1)

def fast_flag(frame):
    return (frame["temperature_c"] >= 80) & (frame["catalyst"] == "Cat-A")

t0 = time.perf_counter(); a = slow_flag(df); t1 = time.perf_counter()
b = fast_flag(df); t2 = time.perf_counter()
print("apply     :", round((t1 - t0) * 1000, 2), "ms")
print("vectorized:", round((t2 - t1) * 1000, 2), "ms")
print("結果一致:", bool((a.fillna(False) == b.fillna(False)).all()))


### 出力の読み方

- 2つの時間（ミリ秒）を比べると、**ベクトル化の方が速い**はずです。420行では差は小さくても、数十万行では体感が大きく変わります。
- `結果一致: True`は、2つの書き方が**同じ答え**を出した確認。速く書いても結果が同じであることを、必ず検証します。
- 教訓：`apply`が必要な場面もありますが、まず「列演算で書けないか」を考える習慣が、速く読みやすいコードにつながります。


## よくある誤り

- 列の単位や定義を確認せず計算する
- 行ごとのapplyを多用して遅く読みにくくする
- 件数が極端に少ない群の平均を強く信じる

## SELF-STUDY（任意・30〜60分）

- 触媒×溶媒の件数・平均収率・標準偏差をpivot_tableで作る
- applyとベクトル化の実行時間を比較し、差をm%で記録する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. shapeの2つの数は何か
2. method chainingの利点と注意点は何か
3. applyよりベクトル化を選ぶ理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
